# Intel RealSense D455 Camera Calibration
The following script demonstrates how to perform intrinsic calibration of the RGB camera on an Intel RealSense D455 using an asymmetric or symmetric checkerboard pattern.

**Prerequisites:**
- Print a checkerboard pattern.
- Update the `CHECKERBOARD` dimensions in the code to match your printed pattern (number of inner corners, typically `(rows-1, cols-1)`).
- Press **'c'** to capture a frame for calibration when the checkerboard is correctly detected (lines are drawn). Capture around 15-20 frames at different angles and distances.
- Press **'q'** to quit the capture loop and compute the calibration matrix.

In [34]:
import pyrealsense2 as rs
import numpy as np
import cv2

def compute_desk_transform(points_3d):
    """
    Computes the 4x4 Transformation Matrix from Camera to Desk.
    Assumes points are: [Top-Left, Top-Right, Bottom-Right, Bottom-Left]
    """
    p0, p1, p2, p3 = points_3d
    
    # 1. Define Translation (Origin is Point 0 / Top-Left)
    origin = p0
    
    # 2. Define X-axis (From Point 0 to Point 1 / Top-Right)
    x_axis = p1 - p0
    x_axis = x_axis / np.linalg.norm(x_axis)
    
    # 3. Define temporary Y-axis (From Point 0 to Point 3 / Bottom-Left)
    y_temp = p3 - p0
    
    # 4. Define Z-axis (Cross product of X and Y_temp gives the normal to the desk)
    z_axis = np.cross(x_axis, y_temp)
    z_axis = z_axis / np.linalg.norm(z_axis)
    
    # 5. Define true orthogonal Y-axis
    y_axis = np.cross(z_axis, x_axis)
    y_axis = y_axis / np.linalg.norm(y_axis)
    
    # 6. Construct 4x4 Transformation Matrix
    rotation_matrix = np.column_stack((x_axis, y_axis, z_axis))
    
    transform_matrix = np.eye(4)
    transform_matrix[:3, :3] = rotation_matrix
    transform_matrix[:3, 3] = origin
    
    return transform_matrix

def main():
    # 1. Configure depth and color streams
    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)

    profile = pipeline.start(config)

    # Align depth frame to color frame
    align_to = rs.stream.color
    align = rs.align(align_to)
    intrinsics = profile.get_stream(rs.stream.color).as_video_stream_profile().get_intrinsics()

    # 2. Setup ArUco Detector for individual markers
    try:
        # Newer OpenCV 4.7+
        aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
        detector_params = cv2.aruco.DetectorParameters()
        detector = cv2.aruco.ArucoDetector(aruco_dict, detector_params)
        OPENCV_OLD = False
    except AttributeError:
        # Older OpenCV versions
        aruco_dict = cv2.aruco.Dictionary_get(cv2.aruco.DICT_4X4_50)
        detector_params = cv2.aruco.DetectorParameters_create()
        OPENCV_OLD = True

    # Expected ArUco Corner IDs for the 4 corners of the desk
    # TL, TR, BR, BL
    expected_ids = [0, 1, 2, 3] 
    
    print(f"Looking for ArUco marker IDs {expected_ids}...")

    try:
        while True:
            frames = pipeline.wait_for_frames()
            aligned_frames = align.process(frames)

            color_frame = aligned_frames.get_color_frame()
            depth_frame = aligned_frames.get_depth_frame()

            if not color_frame or not depth_frame:
                continue

            color_image = np.asanyarray(color_frame.get_data())
            gray_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2GRAY)

            # 3. Detect ArUco Markers
            if OPENCV_OLD:
                corners, ids, rejected = cv2.aruco.detectMarkers(
                    gray_image, aruco_dict, parameters=detector_params
                )
            else:
                corners, ids, rejected = detector.detectMarkers(gray_image)

            if ids is not None and len(ids) > 0:
                cv2.aruco.drawDetectedMarkers(color_image, corners, ids)
                
                detected_ids = ids.flatten().tolist()
                
                # Check if all 4 required markers are in the frame
                if all(req_id in detected_ids for req_id in expected_ids):
                    points_3d = []
                    valid_depth = True

                    # Extract centers and 3D coordinates in the correct order
                    for req_id in expected_ids:
                        idx = detected_ids.index(req_id)
                        marker_corners = corners[idx][0]
                        
                        # Calculate the center pixel of the marker
                        center_x = int(np.mean(marker_corners[:, 0]))
                        center_y = int(np.mean(marker_corners[:, 1]))
                        
                        cv2.circle(color_image, (center_x, center_y), 5, (0, 0, 255), -1)

                        # Get depth and deproject to 3D
                        depth = depth_frame.get_distance(center_x, center_y)
                        if depth == 0:
                            valid_depth = False
                            break
                        
                        point_3d = rs.rs2_deproject_pixel_to_point(intrinsics, [center_x, center_y], depth)
                        points_3d.append(np.array(point_3d))

                    if valid_depth and len(points_3d) == 4:
                        T_camera_to_desk = compute_desk_transform(points_3d)
                        
                        print("\n=== CALIBRATION SUCCESSFUL ===")
                        print("Transformation Matrix (Camera to Desk):")
                        print(np.round(T_camera_to_desk, 4))
                        
                        np.save('camera_to_desk_transform.npy', T_camera_to_desk)
                        print("\nSaved as 'camera_to_desk_transform.npy'.")
                        
                        cv2.imshow('ArUco Calibration', color_image)
                        cv2.waitKey(2000) 
                        break

            cv2.imshow('ArUco Calibration', color_image)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    finally:
        pipeline.stop()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Looking for ArUco marker IDs [0, 1, 2, 3]...


# Step 2: OpenManipulator-X Control

Now that we have the camera-to-desk transformation matrix, we can convert any 3D point seen by the camera into a "Desk Coordinate". 

To move the OpenManipulator-X, you need to:
1. Load the `camera_to_desk_transform.npy`.
2. Convert your target's **Camera coordinates** -> **Desk coordinates**.
3. Measure the physical offset from your Desk Origin (Marker 0) to the **Robot's Base** to get **Robot coordinates**.
4. Send those target XYZ coordinates to the robot.

*Note: Depending on your setup, you might be controlling the OpenManipulator using **ROS (Robot Operating System)** or directly via Python using the **Dynamixel SDK**.*

In [35]:
import numpy as np

# Note: You will need to install the Dynamixel SDK for Python
# Run in terminal: pip install dynamixel-sdk
import dynamixel_sdk as dxl

# ---------------------------------------------------------
# 1. CAMERA TO ROBOT COORDINATE CONVERSION
# ---------------------------------------------------------
# Load the calibration transform we generated earlier
T_camera_to_desk = np.load('camera_to_desk_transform.npy')

# Example: Pretend the camera detected an object at this 3D point (X, Y, Z, 1) in meters
camera_target_point = np.array([0.1, 0.05, 0.7, 1.0])

# Transform from Camera space down to Desk space (Origin is Marker 0)
desk_coord = np.linalg.inv(T_camera_to_desk).dot(camera_target_point)
print(f"Target on Desk (X, Y, Z in meters): {desk_coord[:3]}")

# Measure the physical distance from your Desk Origin (Top-Left marker) to the Robot Base
# Assuming the robot is glued to the desk 20 cm to the right (+X) and 30 cm down (+Y)
robot_offset_from_origin_m = np.array([0.20, 0.30, 0.00]) 

# Final Coordinates relative to the OpenManipulator-X Base
target_x, target_y, target_z = desk_coord[:3] - robot_offset_from_origin_m
print(f"Target for OpenManipulator (X, Y, Z) meters: {target_x:.3f}, {target_y:.3f}, {target_z:.3f}")


# ---------------------------------------------------------
# 2. OPENMANIPULATOR DIRECT PYTHON CONTROL (Dynamixel SDK)
# ---------------------------------------------------------
# OpenManipulator mostly uses XM430-W350-T motors.
# Joint IDs usually: 11, 12, 13, 14. Gripper: 15.
PROTOCOL_VERSION = 2.0
BAUDRATE = 1000000
DEVICENAME = 'COM3'  # Windows: Check Device Manager for your U2D2 port. Linux: '/dev/ttyUSB0'

# Dynamixel Control Table Addresses
ADDR_TORQUE_ENABLE = 64
ADDR_GOAL_POSITION = 116
ADDR_PRESENT_POSITION = 132
TORQUE_ENABLE = 1
TORQUE_DISABLE = 0

# IDs
JOINT_IDS = [11, 12, 13, 14]
GRIPPER_ID = 15

def calculate_inverse_kinematics(x, y, z):
    """
    Inverse Kinematics (IK) Solver maps the 3D XYZ target to 4 motor angles.
    Because the OpenManipulator is a 4-DOF planar arm mounted on a rotating base,
    you must mathematically convert (X, Y, Z) to Joint Angles (theta1, theta2, theta3, theta4).
    
    Implementing a full custom IK solver here requires exact link lengths.
    Common libraries for IK in Python: `Robotics Toolbox`, `ikpy`.
    
    For now, this returns a safe dummy "Home" position in raw motor ticks (0-4095 range)
    representing ~ (0, 0, 0, 0) degrees.
    """
    print(f"\n[IK Solver] Calculating angles for XYZ: ({x:.3f}, {y:.3f}, {z:.3f})")
    # Return placeholder raw positions (2048 is centered for X-Series dynamixels)
    return [2048, 2048, 2048, 2048]

def move_robot_to_xyz(x, y, z):
    # 1. Run IK
    goal_positions = calculate_inverse_kinematics(x, y, z)
    
    # 2. Initialize PortHandler and PacketHandler
    portHandler = dxl.PortHandler(DEVICENAME)
    packetHandler = dxl.PacketHandler(PROTOCOL_VERSION)
    
    # Open port & Set baudrate
    if portHandler.openPort() and portHandler.setBaudRate(BAUDRATE):
        print(f"Succeeded to open the port {DEVICENAME}")
    else:
        print("Failed to open the port. Check your COM/ttyUSB settings.")
        return

    # Enable Torque for all joints
    for dxl_id in JOINT_IDS:
        packetHandler.write1ByteTxRx(portHandler, dxl_id, ADDR_TORQUE_ENABLE, TORQUE_ENABLE)
        
    print("Moving robot...")
    
    # Write Goal Positions
    for i, dxl_id in enumerate(JOINT_IDS):
        # Position is a 4-byte value
        param_goal_position = [
            dxl.DXL_LOBYTE(dxl.DXL_LOWORD(goal_positions[i])),
            dxl.DXL_HIBYTE(dxl.DXL_LOWORD(goal_positions[i])),
            dxl.DXL_LOBYTE(dxl.DXL_HIWORD(goal_positions[i])),
            dxl.DXL_HIBYTE(dxl.DXL_HIWORD(goal_positions[i]))
        ]
        
        packetHandler.writeTxRx(portHandler, dxl_id, ADDR_GOAL_POSITION, 4, param_goal_position)
        
    print("Movement command sent.")
    
    # Disable torque when done (optional, depending on application)
    # for dxl_id in JOINT_IDS:
    #     packetHandler.write1ByteTxRx(portHandler, dxl_id, ADDR_TORQUE_ENABLE, TORQUE_DISABLE)
        
    portHandler.closePort()

# Uncomment to actually run the move command! Ensure your DEVICENAME / COM port is correct.
# move_robot_to_xyz(target_x, target_y, target_z)

Target on Desk (X, Y, Z in meters): [0.27953661 0.07727222 0.14159755]
Target for OpenManipulator (X, Y, Z) meters: 0.080, -0.223, 0.142
